# 02. Señales básicas: impulso, escalón, rampa, exponencial

**Objetivos de aprendizaje.** Al terminar podrás:

1. Construir **paso a paso** (con ciclos) el impulso $\delta[n]$, el escalón $u[n]$, la rampa $r[n]$ y la exponencial $a^n$.
2. Desplazarlas ($\delta[n-n_0]$, $u[n-n_0]$) entendiendo hacia dónde se mueven.
3. Relacionar las señales básicas entre sí.
4. Comprobar que **toda señal discreta es una suma de impulsos desplazados y escalados** — la base de la convolución.

**Requisito:** notebooks 00 y 01.

In [ ]:
# Celda de preparación. Ejecútala antes que cualquier otra (Shift + Enter).
import sys
import os

# Si estás en Google Colab, esta parte descarga el material del curso.
# En tu computadora no hace nada.
URL_REPO = "https://github.com/TU_USUARIO/TU_REPOSITORIO.git"
if "google.colab" in sys.modules and not os.path.exists("../funciones_pds"):
    if not os.path.exists("/content/pds"):
        os.system(f"git clone -q {URL_REPO} /content/pds")
    os.chdir("/content/pds/notebooks")   # así funcionan las rutas "../datos/..."

sys.path.append("..")            # para que Python encuentre la carpeta funciones_pds

import numpy as np               # cálculos con arreglos
import matplotlib.pyplot as plt  # gráficas
import funciones_pds as pds      # la biblioteca del curso

plt.rcParams["figure.figsize"] = (9, 3)   # tamaño por defecto de las figuras
print("Bibliotecas cargadas. Ya puedes continuar.")

## 1. El impulso unitario $\delta[n]$

$$\delta[n] = \begin{cases} 1, & n = 0 \\ 0, & n \ne 0 \end{cases}$$

Es la señal más sencilla: **un solo 1** en $n=0$. A diferencia del impulso analógico (delta de Dirac), no tiene
nada de "infinito": es simplemente el número 1 en una muestra.

### Construcción paso a paso (para $n = -5, \dots, 5$)

In [ ]:
# Paso 1: el eje de índices (recuerda: arange NO incluye el último, por eso 5 + 1)
n = np.arange(-5, 5 + 1)
print("n =", n)

In [ ]:
# Paso 2: una señal de ceros del MISMO tamaño que n
x = np.zeros(len(n))
print("x =", x)

In [ ]:
# Paso 3: recorremos cada posición; donde el ÍNDICE DE TIEMPO n vale 0, ponemos 1
for i in range(len(n)):
    if n[i] == 0:
        x[i] = 1.0
        print(f"   el instante n = 0 está en la posición i = {i} del arreglo")
print("x =", x)

In [ ]:
pds.graficar_discreta(n, x, titulo="δ[n]")
plt.show()

Estos pasos están empaquetados en la función `pds.impulso(n_inicio, n_fin, n0)`.

In [ ]:
import inspect
print(inspect.getsource(pds.impulso))

## 2. Desplazamiento: $\delta[n - n_0]$

$\delta[n - n_0]$ vale 1 cuando $n - n_0 = 0$, es decir en $n = n_0$.

> **Error frecuente:** $\delta[n-3]$ está en $n = +3$ (a la **derecha**), y $\delta[n+3]$ en $n = -3$ (a la **izquierda**).
> El signo "parece" contrario: piensa *"¿dónde se hace cero el argumento?"*.

In [ ]:
figura, ejes = plt.subplots(1, 3, figsize=(13, 3))

n, x = pds.impulso(-5, 5, n0=0)
pds.graficar_discreta(n, x, titulo="δ[n]", ax=ejes[0])

n, x = pds.impulso(-5, 5, n0=3)
pds.graficar_discreta(n, x, titulo="δ[n - 3]  (retrasado)", ax=ejes[1])

n, x = pds.impulso(-5, 5, n0=-3)
pds.graficar_discreta(n, x, titulo="δ[n + 3]  (adelantado)", ax=ejes[2])

plt.tight_layout()
plt.show()

## 3. El escalón unitario $u[n]$

$$u[n] = \begin{cases} 1, & n \ge 0 \\ 0, & n < 0 \end{cases}$$

Representa "encender" algo en $n=0$ y dejarlo encendido. El paso clave es **el mismo** que en el impulso,
sólo cambia la condición: `n[i] == 0` → `n[i] >= 0`.

In [ ]:
n = np.arange(-5, 11)
x = np.zeros(len(n))
for i in range(len(n)):
    if n[i] >= 0:
        x[i] = 1.0

pds.graficar_discreta(n, x, titulo="u[n] (construido a mano)")
plt.show()

n2, x2 = pds.escalon(-5, 10)
pds.comprobar_iguales(x, x2, "nuestro escalón coincide con pds.escalon")

## 4. Rampa y exponencial

$$r[n] = n\,u[n] \qquad\qquad x[n] = A\,a^n$$

La exponencial discreta tiene 4 comportamientos según el valor de $a$:

In [ ]:
n, r = pds.rampa(-3, 8)
pds.graficar_discreta(n, r, titulo="r[n] = n·u[n]")
plt.show()

In [ ]:
figura, ejes = plt.subplots(2, 2, figsize=(11, 6))
casos = [(0.8, "0 < a < 1: decrece"), (-0.8, "-1 < a < 0: decrece alternando signo"),
         (1.2, "a > 1: crece"), (-1.2, "a < -1: crece alternando signo")]

for i in range(len(casos)):
    a, descripcion = casos[i]
    n, x = pds.exponencial(0, 15, a)
    eje = ejes[i // 2, i % 2]           # fila y columna de la subgráfica
    pds.graficar_discreta(n, x, titulo=f"a = {a}: {descripcion}", ax=eje)

plt.tight_layout()
plt.show()

## 5. Relaciones entre las señales básicas

$$\delta[n] = u[n] - u[n-1] \qquad\qquad u[n] = \sum_{k=0}^{\infty} \delta[n-k] \qquad\qquad r[n] = n\,u[n]$$

En lugar de creerlo, comprobémoslo numéricamente.

In [ ]:
n, u = pds.escalon(-5, 10, n0=0)
n, u_1 = pds.escalon(-5, 10, n0=1)
n, delta = pds.impulso(-5, 10)

diferencia = u - u_1
pds.comprobar_iguales(diferencia, delta, "δ[n] = u[n] - u[n-1]")

In [ ]:
# u[n] como suma de impulsos δ[n] + δ[n-1] + δ[n-2] + ... (hasta el final de nuestro eje)
suma = np.zeros(len(n))
for k in range(0, 11):
    _, delta_k = pds.impulso(-5, 10, n0=k)
    suma = suma + delta_k

pds.comprobar_iguales(suma, u, "u[n] = Σ δ[n-k]")

### Pulso rectangular

Restando dos escalones obtenemos un pulso de duración $L$: $\;p[n] = u[n] - u[n-L]$.

In [ ]:
L = 4
n, u0 = pds.escalon(-3, 10, n0=0)
n, uL = pds.escalon(-3, 10, n0=L)
pulso = u0 - uL

figura, ejes = plt.subplots(1, 3, figsize=(13, 3))
pds.graficar_discreta(n, u0, titulo="u[n]", ax=ejes[0])
pds.graficar_discreta(n, uL, titulo=f"u[n-{L}]", ax=ejes[1])
pds.graficar_discreta(n, pulso, titulo=f"u[n] - u[n-{L}]", ax=ejes[2])
plt.tight_layout()
plt.show()

## 6. Toda señal es una suma de impulsos

Cualquier señal discreta se puede escribir como:

$$x[n] = \sum_{k} x[k]\,\delta[n-k]$$

Leámoslo despacio: "en cada posición $k$ ponemos un impulso ($\delta[n-k]$) y lo multiplicamos por el valor
que la señal tiene ahí ($x[k]$)". Sumando todos esos impulsos recuperamos la señal.

**Ejemplo:** $x[n] = \{\, 2,\; \underset{\uparrow}{-1},\; 3,\; 0.5 \,\}$ (la flecha marca $n=0$), es decir:

$$x[n] = 2\,\delta[n+1] - 1\,\delta[n] + 3\,\delta[n-1] + 0.5\,\delta[n-2]$$

In [ ]:
n = np.arange(-1, 3)
x = np.array([2.0, -1.0, 3.0, 0.5])

componentes = pds.descomponer_en_impulsos(n, x)

figura, ejes = plt.subplots(1, len(componentes) + 1, figsize=(15, 3), sharey=True)
for i in range(len(componentes)):
    k = n[i]
    pds.graficar_discreta(n, componentes[i], titulo=f"{x[i]}·δ[n-({k})]", ax=ejes[i])

y = pds.reconstruir_desde_impulsos(componentes)
pds.graficar_discreta(n, y, titulo="SUMA = x[n]", ax=ejes[-1], color="C3")
plt.tight_layout()
plt.show()

pds.comprobar_iguales(y, x, "la suma de impulsos reconstruye la señal")

> **¿Por qué es tan importante?** Si sabemos cómo responde un sistema a **un** impulso ($h[n]$, la *respuesta al impulso*),
> y el sistema es lineal e invariante, entonces sabemos cómo responde a **cualquier** señal: basta sumar las respuestas
> a cada impulso. Eso es la **convolución** (notebook 06).

## 7. Construir señales por tramos combinando señales básicas

Ejemplo: $x[n] = 0.9^n\,\big(u[n] - u[n-10]\big)$ — una exponencial que sólo dura 10 muestras.

In [ ]:
n, e = pds.exponencial(-3, 15, a=0.9)
n, u0 = pds.escalon(-3, 15, n0=0)
n, u10 = pds.escalon(-3, 15, n0=10)

ventana = u0 - u10            # pulso de n = 0 a n = 9
x = e * ventana               # producto muestra a muestra

pds.graficar_discreta(n, x, titulo="x[n] = 0.9ⁿ·(u[n] - u[n-10])")
plt.show()

## Comprueba tu comprensión

1. ¿Dónde está el 1 de $\delta[n+2]$?
2. ¿Qué señal es $u[n] - u[n-1]$? ¿Y $u[n+2] - u[n-3]$ (cuántas muestras dura y dónde empieza)?
3. Escribe $x[n] = \{\underset{\uparrow}{1}, 4, 2\}$ como suma de impulsos.

<details><summary>Ver respuestas</summary>

1. En $n=-2$.
2. $\delta[n]$. Un pulso de 5 muestras, de $n=-2$ a $n=2$.
3. $x[n] = \delta[n] + 4\delta[n-1] + 2\delta[n-2]$.
</details>